# Why MLOps Matters More Than Your Model

Companion notebook for the article. Run all cells to reproduce every number. Needs NumPy and scikit-learn.

From [*Math for ML Engineers: From Backprop to RLHF*](https://mikaelyemane.github.io/ml-math-code/) by Mikael Yemane.

In [1]:
%pip install -q numpy scikit-learn

## Why ML systems fail silently

In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score

rng = np.random.default_rng(0)
n = 20_000
latency_ms = rng.gamma(2.0, 150.0, n)
retries = rng.poisson(1.0, n)
logit = 0.004 * latency_ms + 0.8 * retries - 3.0
y = rng.random(n) < 1 / (1 + np.exp(-logit))
X = np.c_[latency_ms, retries]

model = LogisticRegression(max_iter=1000).fit(X[:10_000], y[:10_000])
X_new = X[10_000:].copy()
X_new[:, 0] /= 1000                      # upstream switches ms -> s

for name, Xt in [("before", X[10_000:]), ("after", X_new)]:
    p = model.predict(Xt)
    print(name, "accuracy", round(model.score(Xt, y[10_000:]), 3),
          "recall", round(recall_score(y[10_000:], p), 3),
          "flagged", round(p.mean(), 3))

before accuracy 0.762 recall 0.406 flagged 0.181
after accuracy 0.709 recall 0.051 flagged 0.018


## Five practices that catch it

In [3]:
import numpy as np

def psi(expected, actual, bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected)
    a = np.histogram(actual, edges)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

## Reproduce the numbers quoted in the text

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
gbm = HistGradientBoostingClassifier(random_state=0).fit(X[:10_000], y[:10_000])
for name, Xt in [("before", X[10_000:]), ("after", X_new)]:
    print("gradient boosting", name, "recall", round(recall_score(y[10_000:], gbm.predict(Xt)), 3))
print("share of requests that fail:", round(y[10_000:].mean(), 3))
print("PSI on latency, two healthy halves:", round(psi(X[:10_000, 0], X[10_000:, 0]), 3))
print("PSI on latency, after ms -> s:", round(psi(X[:10_000, 0], X_new[:, 0]), 1))
print("converted latencies below 1:", f"{np.mean(X_new[:, 0] < 1):.0%}")

gradient boosting before recall 0.374
gradient boosting after recall 0.051
share of requests that fail: 0.304
PSI on latency, two healthy halves: 0.002
PSI on latency, after ms -> s: 12.4
converted latencies below 1: 99%
